In [1]:
# Nomes das colunas de interesse nos dataframes
COL_ID_TRECHO = 'Identificador do processo de viagem' 
COL_ID_TRECHO_SPACED = 'Identificador do processo de viagem ' 
COL_ORIGEM = 'Origem - Cidade'
COL_DESTINO = 'Destino - Cidade'

from viagens_dowload import Viagens # Importa a classe Viagens do módulo viagens_dowload
ano = 2023
vi = Viagens(ano)

In [2]:
# faz download dos dados de viagens a serviço e carrega os dataframes
# viagem_df, trecho_df, passagem_df = vi.pegarViagens()

# carregar arquivos

In [2]:
#se já tiver os csvs baixados, só carrega os dataframes
viagem_df, trecho_df, passagem_df = vi.carregar_csvs()

✅ 2023_Passagem.csv carregado com 390728 linhas (sem cabeçalho).
✅ 2023_Trecho.csv carregado com 1817649 linhas (sem cabeçalho).
✅ 2023_Viagem.csv carregado com 832499 linhas (sem cabeçalho).


In [3]:
trecho_df.columns

Index(['Identificador do processo de viagem ', 'Número da Proposta (PCDP)',
       'Sequência Trecho', 'Origem - Data', 'Origem - País', 'Origem - UF',
       'Origem - Cidade', 'Destino - Data', 'Destino - País', 'Destino - UF',
       'Destino - Cidade', 'Meio de transporte', 'Número Diárias', 'Missao?'],
      dtype='object', name=0)

In [5]:
# --- CÉLULA 2  Filtragem, Limpeza e Criação da Chave ---

# --- 0. FILTRO INICIAL CRÍTICO: Situação = "Realizada" ---
# A coluna 'Situação' está no passagem_df.
VIAGEM_STATUS_FILTRO = "Realizada"

# Filtra o DataFrame de Passagens
linhas_antes = len(passagem_df)
passagem_df = passagem_df[passagem_df['Situação'] == VIAGEM_STATUS_FILTRO].copy()
linhas_depois = len(passagem_df)

print(f"✅ Filtro inicial aplicado: Mantidas {linhas_depois} viagens '{VIAGEM_STATUS_FILTRO}'.")
print(f"   ({linhas_antes - linhas_depois} viagens não realizadas foram descartadas).")


# 1. Limpeza do trecho_df
# Renomeia a coluna com erro de espaço para garantir o merge
if COL_ID_TRECHO_SPACED in trecho_df.columns:
    trecho_df = trecho_df.rename(columns={COL_ID_TRECHO_SPACED: COL_ID_TRECHO})
    print(f"✅ Coluna '{COL_ID_TRECHO_SPACED}' renomeada para '{COL_ID_TRECHO}'.")

# 2. Garante tipos de dados e limpa valores
for df in [passagem_df, trecho_df, viagem_df]:
    if COL_ID_TRECHO in df.columns:
        df[COL_ID_TRECHO] = df[COL_ID_TRECHO].astype(str).str.strip()

if COL_ORIGEM in trecho_df.columns and COL_DESTINO in trecho_df.columns:
    trecho_df[COL_ORIGEM] = trecho_df[COL_ORIGEM].astype(str).str.strip()
    trecho_df[COL_DESTINO] = trecho_df[COL_DESTINO].astype(str).str.strip()
    print("✅ Normalização das chaves e cidades concluída.")
else:
    print(f"❌ Erro: Colunas de origem/destino ({COL_ORIGEM}/{COL_DESTINO}) não encontradas no trecho_df.")


# 3. CRIAÇÃO DA CHAVE TEMPORÁRIA
trecho_df['Cod Trecho Temp'] = trecho_df[COL_ORIGEM] + '_' + trecho_df[COL_DESTINO]
trecho_df['Cod Trecho Temp'] = trecho_df['Cod Trecho Temp'].str.strip()

print("✅ Chave 'Cod Trecho Temp' criada.")

✅ Filtro inicial aplicado: Mantidas 815707 viagens 'Realizada'.
   (16792 viagens não realizadas foram descartadas).
✅ Coluna 'Identificador do processo de viagem ' renomeada para 'Identificador do processo de viagem'.
✅ Normalização das chaves e cidades concluída.
✅ Chave 'Cod Trecho Temp' criada.


# calculando distancias

In [6]:
import pandas as pd

# --- CORREÇÕES MANUAIS DE ORTOGRAFIA ---
# Mapeamento dos nomes incorretos para os nomes reconhecidos pelo Nominatim/API.

# --- MAPEAMENTO COMPLETO DE CORREÇÕES (DIGITAÇÃO + ENCODING) ---

correcoes = {
    # 1. ERROS DE DIGITAÇÃO / MISSPELINGS (Anteriores)
    'Palma de Malorca': 'Palma de Mallorca',
    'Wroc?aw': 'Wroclaw', # Usando a versão sem caracteres especiais para máxima compatibilidade
    'Serinagar': 'Srinagar',
    'Islascanarias': 'Ilhas Canárias',
    'Bangcoc': 'Bangkok',
    
    # 2. CORREÇÕES DE CODIFICAÇÃO (Strings Corrompidas lidas do CSV)
    'AnaheimCA - CalifÃ³rnia': 'AnaheimCA - Califórnia',
    'NiterÃ³i': 'Niterói',
    'MacapÃ¡ - ArquipÃ©lago do Bailique': 'Macapá - Arquipélago do Bailique',
    'MacapÃ¡': 'Macapá',
    'VitÃ³ria': 'Vitória',
    'UÃ¡di Natrum': 'Uádi Natrum',
    'UberlÃ¢ndia': 'Uberlândia',
    'TÃ³quio': 'Tóquio',
    'TetuÃ£o': 'Tetuão',
    'SÃ£o Paulo': 'São Paulo',
    'SÃ£o LuÃ\xads': 'São Luís',
    'Santa Helena do UairÃ©n': 'Santa Helena do Uairén',
    'Cartagena das Ã\x8dndias': 'Cartagena das Índias',
    'CantÃ£o de Soleura': 'Cantão de Soleura',
    'BrasÃ\xadlia': 'Brasília',
    'BogotÃ¡': 'Bogotá',
    'BelÃ©m': 'Belém',
    'AssunÃ§Ã£o': 'Assunção',
    'GoiÃ¢nia': 'Goiânia',
    'Foz do IguaÃ§u': 'Foz do Iguaçu',
    'Fort Benning South - GeÃ³rgia': 'Fort Benning South - Geórgia',
    'FlorianÃ³polis': 'Florianópolis',
    'CosenÃ§a': 'Cosença',
    'Cidade do PanamÃ¡': 'Cidade do Panamá',
    'B?dlewo': 'Bydlewo', # Assumindo a correção para a cidade polonesa
    'AnaheimCA - Califórnia': 'Anaheim - Califórnia ',
    'Marraqueche': 'Marrakech',
    'Ponta Arenas': 'Punta Arenas',
    'Santa Helena do Uairén': 'Santa Elena de Uairén',
    # novas
    'AssunÃ§Ã£o': 'Assunção',
    'NiterÃ³i': 'Niterói',
    'MacapÃ¡ - ArquipÃ©lago do Bailique': 'Macapá - Arquipélago do Bailique',
    'MacapÃ¡': 'Macapá',
    'VitÃ³ria': 'Vitória',
    'UÃ¡di Natrum': 'Uádi Natrum',
    'UberlÃ¢ndia': 'Uberlândia',
    'TÃ³quio': 'Tóquio',
    'TetuÃ£o': 'Tetuão',
    'SÃ£o Paulo': 'São Paulo',
    'SÃ£o LuÃ\xads': 'São Luís',
    'Santa Helena do UairÃ©n': 'Santa Helena do Uairén',
    'Cartagena das Ã\x8dndias': 'Cartagena das Índias',
    'CantÃ£o de Soleura': 'Cantão de Soleura',
    'BrasÃ\xadlia': 'Brasília',
    'BogotÃ¡': 'Bogotá',
    'BelÃ©m': 'Belém',
    'GoiÃ¢nia': 'Goiânia',
    'Foz do IguaÃ§u': 'Foz do Iguaçu',
    'Fort Benning South - GeÃ³rgia': 'Fort Benning South - Geórgia',
    'FlorianÃ³polis': 'Florianópolis',
    'CosenÃ§a': 'Cosença',
    'Cidade do PanamÃ¡': 'Cidade do Panamá',
    #######################
    'Palma de Malorca': 'Palma de Mallorca',
    'Wroc?aw': 'Wroclaw', 
    'Serinagar': 'Srinagar',
    'Islascanarias': 'Ilhas Canárias',
    'Bangcoc': 'Bangkok',
    
    # 2. CORREÇÕES DE CODIFICAÇÃO (Strings Corrompidas lidas do CSV):
    # Estas correções são vitais e resolvem a maioria das 164 falhas.
    'AssunÃ§Ã£o': 'Assunção',
    'NiterÃ³i': 'Niterói',
    'MacapÃ¡ - ArquipÃ©lago do Bailique': 'Macapá - Arquipélago do Bailique',
    'MacapÃ¡': 'Macapá',
    'VitÃ³ria': 'Vitória',
    'UÃ¡di Natrum': 'Uádi Natrum',
    'UberlÃ¢ndia': 'Uberlândia',
    'TÃ³quio': 'Tóquio',
    'TetuÃ£o': 'Tetuão',
    'SÃ£o Paulo': 'São Paulo',
    'SÃ£o LuÃ\xads': 'São Luís',
    'Santa Helena do UairÃ©n': 'Santa Helena do Uairén',
    'Cartagena das Ã\x8dndias': 'Cartagena das Índias',
    'CantÃ£o de Soleura': 'Cantão de Soleura',
    'BrasÃ\xadlia': 'Brasília',
    'BogotÃ¡': 'Bogotá',
    'BelÃ©m': 'Belém',
    'GoiÃ¢nia': 'Goiânia',
    'Foz do IguaÃ§u': 'Foz do Iguaçu',
    'Fort Benning South - GeÃ³rgia': 'Fort Benning South - Geórgia',
    'FlorianÃ³polis': 'Florianópolis',
    'CosenÃ§a': 'Cosença',
    'Cidade do PanamÃ¡': 'Cidade do Panamá',
    'AnaheimCA - CalifÃ³rnia': 'AnaheimCA - Califórnia',
    'Honj?': 'Honjo',
    'B?dlewo': 'Bydlewo',
    # 1. ERROS DE DIGITAÇÃO / MISSPELINGS (Anteriores)
    'Palma de Malorca': 'Palma de Mallorca',
    'Wroc?aw': 'Wroclaw', 
    'Serinagar': 'Srinagar',
    'Islascanarias': 'Ilhas Canárias',
    'Bangcoc': 'Bangkok',
    
    # 2. NOMES CORROMPIDOS / TRADUÇÕES EXPLICITAMENTE CORRIGIDAS (NOVOS)
    'Cantão de Soleura': 'Solothurn, Suíça', # Nome internacionalmente reconhecido
    'Cartagena das Índias': 'Cartagena, Colômbia', # Adicionando país para forçar a localização
    'Cosença': 'Cosenza', # Correção do Português para o Italiano
    'Distrito de Yguazu': 'Minga Guazú, Paraguai', # Nome de localidade reconhecido
    'Fort Benning South - Geórgia': 'Fort Moore, GA, EUA', # Nome atual e localização oficial
    'Ludlow - Massachussets': 'Ludlow, Massachusetts', # Removendo o hífen
    'Macapá - Arquipélago do Bailique': 'Macapá', # Simplificando para a cidade principal
    'Tetuão': 'Tétouan, Marrocos', # Nome internacional + país
    'Uádi Natrum': 'Wadi El Natrun, Egito', # Nome fonético correto
    'Vilamontes': 'Villamontes', # Correção de digitação
    'Bydlewo': 'Bydlewo, Polônia', # Adicionando país
    'Sophia-Atipolis': 'Sophia Antipolis', # Removendo hífen e corrigindo ortografia
    'Itajuipi': 'Itajuípe, Bahia', # Corrigindo e adicionando estado para precisão
    
    # 3. CORREÇÕES DE CODIFICAÇÃO (Mantidas para segurança)
    'AssunÃ§Ã£o': 'Assunção', 'NiterÃ³i': 'Niterói', 'MacapÃ¡': 'Macapá', 'VitÃ³ria': 'Vitória', 
    'UberlÃ¢ndia': 'Uberlândia', 'SÃ£o Paulo': 'São Paulo', 'BrasÃ\xadlia': 'Brasília',
    
    # 'B?dlewo': 'Będlewo, Poland',
    # 'Będlewo':'Bedlewo'
    'Barcelona': 'Barcelona, Espanha',
}

correcoes = dict(sorted(correcoes.items()))  # opcional: ordena alfabeticamente


# 2. Aplicar as Correções Diretamente ao trecho_df (Base de Trechos)
COLUNAS_CIDADE = ['Origem - Cidade', 'Destino - Cidade']

for col in COLUNAS_CIDADE:
    # O .replace() é a forma mais eficiente de aplicar o dicionário de correção.
    trecho_df[col] = trecho_df[col].replace(correcoes)
    
    # Aplica um strip final para limpar espaços remanescentes
    trecho_df[col] = trecho_df[col].astype(str).str.strip()


print("✅ Correções manuais de grafia aplicadas ao DataFrame 'trecho_df'.")



✅ Correções manuais de grafia aplicadas ao DataFrame 'trecho_df'.


In [ ]:
# --- CÉLULA 3: GEOCÓDIGO E CÁLCULO VETORIZADO (VERSÃO ROBUSTA E FINAL) ---
from distanciaCidades2Oficial import Distancia
import pandas as pd
import numpy as np

# 1. Preparação da Classe e Lista de Cidades
geocoder = Distancia() 

# 🚨 CORREÇÃO CRÍTICA DE MERGE: Define a lista de colunas de coordenadas
COORD_COLS_FLUTUANTES = ['Lat_Origem', 'Lon_Origem', 'Lat_Destino', 'Lon_Destino']

# Limpa colunas de coordenadas antigas ANTES do merge
trecho_df = trecho_df.drop(columns=COORD_COLS_FLUTUANTES, errors='ignore')


# Concatena todas as cidades de origem e destino para obter a lista única
cidades_origem = trecho_df[COL_ORIGEM].unique()
cidades_destino = trecho_df[COL_DESTINO].unique()
cidades_trecho_list = np.unique(np.concatenate((cidades_origem, cidades_destino))).tolist()


# 2. Obter Mapa de Coordenadas Híbrido (Gerenciamento de API e Cache)
df_coordenadas_mapa = geocoder.obter_mapa_coordenadas_hibridas(cidades_trecho_list)


# 3. Merge das Coordenadas no DataFrame de Trechos
# Usamos novos nomes de coluna temporários no DF de coordenadas para garantir que não haja conflito
df_coordenadas_origem = df_coordenadas_mapa.rename(
    columns={'Latitude': 'Lat_Origem_TEMP', 'Longitude': 'Lon_Origem_TEMP', 'Cidade': COL_ORIGEM}
)

df_coordenadas_destino = df_coordenadas_mapa.rename(
    columns={'Latitude': 'Lat_Destino_TEMP', 'Longitude': 'Lon_Destino_TEMP', 'Cidade': COL_DESTINO}
)

# Merge para a Origem
trecho_df = pd.merge(
    trecho_df,
    df_coordenadas_origem,
    on=COL_ORIGEM,
    how='left'
)

# Merge para o Destino
trecho_df = pd.merge(
    trecho_df,
    df_coordenadas_destino,
    on=COL_DESTINO,
    how='left'
)

# 4. Renomear as colunas TEMP para os nomes finais
trecho_df = trecho_df.rename(columns={
    'Lat_Origem_TEMP': 'Lat_Origem',
    'Lon_Origem_TEMP': 'Lon_Origem',
    'Lat_Destino_TEMP': 'Lat_Destino',
    'Lon_Destino_TEMP': 'Lon_Destino',
})


# 5. CÁLCULO VETORIZADO DA DISTÂNCIA (GCD)

# GARANTIA DE TIPO CRÍTICA: Converte para float, forçando NaNs se houver dados ruins
for col in COORD_COLS_FLUTUANTES:
    trecho_df[col] = pd.to_numeric(trecho_df[col], errors='coerce')


# FILTRO CRÍTICO: Se qualquer uma das 4 coordenadas for NaN, zera todas as 4 Lat/Lon
linhas_com_nan_coord = trecho_df[COORD_COLS_FLUTUANTES].isna().any(axis=1)
trecho_df.loc[linhas_com_nan_coord, COORD_COLS_FLUTUANTES] = np.nan
linhas_com_nan_coord_count = linhas_com_nan_coord.sum()
print(f"⚠️ Aviso: {linhas_com_nan_coord_count} trechos tiveram coordenadas zeradas por falha no geocoding.")


# Aplica a função Haversine
trecho_df['Distância (GCD)'] = Distancia.calcular_haversine(
    trecho_df['Lat_Origem'], 
    trecho_df['Lon_Origem'], 
    trecho_df['Lat_Destino'], 
    trecho_df['Lon_Destino']
)
# # Converte para Inteiro Anulável (aceita pd.NA)
# trecho_df['Distância (GCD)'] = trecho_df['Distância (GCD)'].astype('Int64')

# FILTRO PÓS-PROCESSAMENTO CRÍTICO
LIMITE_MAXIMO_KM = 21000 
linhas_invalidas = trecho_df['Distância (GCD)'] > LIMITE_MAXIMO_KM
linhas_invalidas_count = linhas_invalidas.sum()
trecho_df.loc[linhas_invalidas, 'Distância (GCD)'] = np.nan 

print(f"⚠️ Aviso: {linhas_invalidas_count} distâncias impossíveis (> {LIMITE_MAXIMO_KM} km) foram convertidas para NaN.")


# 6. Limpeza e Preparação para o Merge Final (Célula 4)
trecho_df = trecho_df.drop(columns=COORD_COLS_FLUTUANTES, errors='ignore')

df_trechos_calculados = trecho_df[['Cod Trecho Temp', 'Distância (GCD)']].copy()

print(f"\n✅ CÁLCULO DE DISTÂNCIA VETORIZADO CONCLUÍDO.")
print(f"Linhas com distância calculada (sem valores absurdos): {df_trechos_calculados['Distância (GCD)'].notna().sum()}")

   -> Cache API carregado com sucesso: 7634 entradas.
✅ Gerenciador de Coordenadas inicializado. Mapa possui 7634 cidades.

🔄 Geocoding API: 1 cidades faltantes encontradas. Iniciando com Rate Limit (1.1s de espera).
   -> Processando: 0/1. Cidade atual: Bydlewo
   -> Cache API carregado com sucesso: 7634 entradas.
✅ Cache FINAL sobrescrito. 7634 cidades totais. Tempo final: 2.22s.
⚠️ Aviso: 3 trechos tiveram coordenadas zeradas por falha no geocoding.
⚠️ Aviso: 0 distâncias impossíveis (> 21000 km) foram convertidas para NaN.

✅ CÁLCULO DE DISTÂNCIA VETORIZADO CONCLUÍDO.
Linhas com distância calculada (sem valores absurdos): 1817646


# vendo cidades problematicas

In [10]:
# --- CÉLULA 3d: DIAGNÓSTICO FINAL DAS CIDADES FALTANTES ---
import numpy as np
import pandas as pd

# 1. Obter a lista completa de cidades únicas do DataFrame trecho_df
cidades_origem = trecho_df[COL_ORIGEM].unique()
cidades_destino = trecho_df[COL_DESTINO].unique()
cidades_trecho_list = np.unique(np.concatenate((cidades_origem, cidades_destino))).tolist()

# 2. Obter o mapa de coordenadas atualizado (que está na memória da instância 'geocoder')
# Usamos o mapa que contém as 7622 cidades válidas.
df_mapa_na_memoria = geocoder.mapa_coordenadas.copy()

# 3. Comparar a lista de cidades requeridas com o mapa na memória (Identifica a diferença)
df_comparacao = pd.Series(cidades_trecho_list).to_frame(name='Cidade')

df_faltantes = pd.merge(
    df_comparacao,
    df_mapa_na_memoria[['Cidade']],
    on='Cidade',
    how='left',
    indicator=True
).query('_merge == "left_only"')

cidades_a_serem_resolvidas = df_faltantes['Cidade'].tolist()

print("--- LISTA FINAL DE CIDADES QUE O SISTEMA NÃO CONSEGUE RESOLVER ---")
print(f"Total de cidades que precisam de coordenadas: {len(cidades_a_serem_resolvidas)}")

if len(cidades_a_serem_resolvidas) > 0:
    for i, cidade in enumerate(cidades_a_serem_resolvidas):
        print(f"{i+1}. {cidade}")
else:
    print("✅ Parabéns! Nenhuma cidade adicional precisa de geocoding.")

--- LISTA FINAL DE CIDADES QUE O SISTEMA NÃO CONSEGUE RESOLVER ---
Total de cidades que precisam de coordenadas: 1
1. Bydlewo


In [11]:
# --- CÉLULA 3c: LISTA DE CIDADES COM FALHA DE COORDENADA ---

# 1. Identifica as linhas onde a Distância (GCD) falhou (NaN)
df_falha_distancia = trecho_df[trecho_df['Distância (GCD)'].isna()].copy()

# 2. Extrai as cidades únicas que causaram a falha (Origem e Destino)
cidades_origem_falhas = df_falha_distancia[COL_ORIGEM].unique()
cidades_destino_falhas = df_falha_distancia[COL_DESTINO].unique()

# 3. Combina as duas listas e remove duplicatas
cidades_com_falha_geocoding = np.unique(np.concatenate((cidades_origem_falhas, cidades_destino_falhas))).tolist()

# 4. Limpa e exibe o resultado
# Filtra para remover strings vazias ou 'nan' textuais
cidades_limpas = [c for c in cidades_com_falha_geocoding if c and str(c).strip().lower() not in ('nan', 'none', '')]

print("--- CIDADES FALTANTES NO MAPA DE COORDENADAS ---")
print(f"Total de {len(cidades_com_falha_geocoding)} cidades únicas causaram falha no cálculo.")
print("Amostra das primeiras 20 cidades que precisam de coordenadas:")

# Exibe as cidades que precisam de coordenadas
for i, cidade in enumerate(cidades_limpas[:20]):
    print(f"- {cidade}")

--- CIDADES FALTANTES NO MAPA DE COORDENADAS ---
Total de 2 cidades únicas causaram falha no cálculo.
Amostra das primeiras 20 cidades que precisam de coordenadas:
- Bydlewo
- Uberlândia


In [12]:
cidades_limpas

['Bydlewo', 'Uberlândia']

## apagando o trecho problematico

In [13]:
import pandas as pd
import numpy as np

# --- REMOÇÃO FINAL DOS TRECHOS QUE FALHARAM (NaN) ---

# 1. Identifica e remove todas as linhas onde a Distância (GCD) é NaN.
# Estas são as linhas que o geocoding não conseguiu resolver (incluindo Bydlewo).
linhas_antes = len(trecho_df)
trecho_df = trecho_df.dropna(subset=['Distância (GCD)']).copy()
linhas_removidas = linhas_antes - len(trecho_df)


# 2. Atualiza a tabela de lookup (df_trechos_calculados)
# Isso é crucial para que a Célula 4 use o lookup correto.
df_trechos_calculados = trecho_df[['Cod Trecho Temp', 'Distância (GCD)']].drop_duplicates().copy()


print("✅ Limpeza de Trechos Faltantes Concluída.")
print(f"Total de trechos removidos (associados às 3 falhas de geocoding): {linhas_removidas}")
print(f"O 'trecho_df' agora está 100% limpo para o merge final.")

✅ Limpeza de Trechos Faltantes Concluída.
Total de trechos removidos (associados às 3 falhas de geocoding): 3
O 'trecho_df' agora está 100% limpo para o merge final.


In [14]:

# --- AGREGAÇÃO POR VIAGEM COMPLETA ---
print("Iniciando agregação por Identificador do processo de viagem...")

viagens_agrupadas = (
    trecho_df.groupby("Identificador do processo de viagem", as_index=False)
    .agg({
        "Distância (GCD)": "sum",   # soma a distância total da viagem
        "Origem - Cidade": "first", # primeira origem
        "Destino - Cidade": "last", # último destino
        # Inclua outras colunas relevantes se necessário
    })
)

# Substituir ou salvar em uma nova variável
df_viagens = viagens_agrupadas.copy()

print(f"✅ Agregação concluída: {len(df_viagens)} viagens únicas.")


Iniciando agregação por Identificador do processo de viagem...
✅ Agregação concluída: 714013 viagens únicas.


In [15]:
# Célula de diagnóstico TEMPORÁRIA (localização correta)

viagem_id_para_verificar = '0000000000019375473'

print(f"--- Verificando o estado da viagem ID {viagem_id_para_verificar} no 'trecho_df' ANTES de remover os NaNs ---")

# Filtra o trecho_df para o ID específico
viagem_antes_do_drop = trecho_df[trecho_df['Identificador do processo de viagem'] == viagem_id_para_verificar]

if viagem_antes_do_drop.empty:
    print("\n❌ A viagem já não existe no 'trecho_df' mesmo antes da limpeza.")
else:
    # Mostra as colunas relevantes
    print(viagem_antes_do_drop[['Origem - Cidade', 'Destino - Cidade', 'Distância (GCD)']])
    
    # Verifica se a distância é NaN/Nula
    if viagem_antes_do_drop['Distância (GCD)'].isna().all():
        print("\n✅ DIAGNÓSTICO CONFIRMADO: Todas as distâncias para esta viagem são NaN/Nulas.")
        print("   É por isso que a viagem está sendo removida na célula de limpeza seguinte.")
        print("   Causa: Falha na geocodificação dos nomes das cidades desta viagem.")
        cidades_problematicas = pd.concat([viagem_antes_do_drop['Origem - Cidade'], viagem_antes_do_drop['Destino - Cidade']]).unique()
        print(f"   Cidades a serem investigadas: {list(cidades_problematicas)}")

--- Verificando o estado da viagem ID 0000000000019375473 no 'trecho_df' ANTES de remover os NaNs ---
        Origem - Cidade Destino - Cidade  Distância (GCD)
1325150     João Pessoa         Valencia      6277.191835
1325151        Valencia      João Pessoa      6277.191835


In [63]:
# --- CÉLULA 3b: VISUALIZAÇÃO DE SANIDADE DA DISTÂNCIA ---

# 1. Seleciona as colunas de interesse e remove NaNs (apenas distâncias calculadas)
df_distancias_validas = trecho_df[[
    'Origem - Cidade', 
    'Destino - Cidade', 
    'Distância (GCD)'
]].dropna(subset=['Distância (GCD)'])

# 2. Filtra distâncias maiores que 1000 km (para focar nas viagens longas de interesse)
df_distancias_validas = df_distancias_validas[df_distancias_validas['Distância (GCD)'] > 1000]

# 3. Ordena pela distância (Decrescente)
df_distancias_validas_ordenado = df_distancias_validas.sort_values(
    by='Distância (GCD)', 
    ascending=False
)

print("--- TOP 10 MAIORES DISTÂNCIAS CALCULADAS (KM) ---")
print("⚠️ O valor máximo deve ser inferior a 21.000 km.")
print(df_distancias_validas_ordenado.head(10).to_string(index=False))

--- TOP 10 MAIORES DISTÂNCIAS CALCULADAS (KM) ---
⚠️ O valor máximo deve ser inferior a 21.000 km.
Origem - Cidade Destino - Cidade  Distância (GCD)
    Passo Fundo          Taizhou     19409.950365
   Porto Alegre           Xangai     19302.422164
   Porto Alegre           Xangai     19302.422164
   Porto Alegre           Xangai     19302.422164
         Xangai     Porto Alegre     19302.422164
   Porto Alegre           Xangai     19302.422164
         Xangai     Porto Alegre     19302.422164
   Porto Alegre           Xangai     19302.422164
   Porto Alegre           Xangai     19302.422164
   Porto Alegre           Xangai     19302.422164


# Merger

In [90]:
# Renomeia as colunas que ficaram com sufixo e que são úteis

# Mapeamento para renomear as colunas
colunas_para_renomear = {
    # Renomeia a coluna PCDP mantida
    'Número da Proposta (PCDP)_x': 'Número da Proposta (PCDP)',
}

df_master = df_master.rename(columns=colunas_para_renomear)

print("✅ Colunas com sufixos (_x) renomeadas.")
print("\n--- Amostra das Colunas Finais ---")
print(df_master.columns.tolist())

✅ Colunas com sufixos (_x) renomeadas.

--- Amostra das Colunas Finais ---
['Identificador do processo de viagem', 'Número da Proposta (PCDP)', 'Cod Trecho Temp', 'Situação', 'Código do órgão superior', 'Nome do órgão superior', 'Código órgão solicitante', 'Nome órgão solicitante', 'CPF viajante', 'Nome', 'Cargo', 'Função', 'Descrição Função', 'Destinos', 'Motivo', 'Origem - Cidade', 'Destino - Cidade', 'Meio de transporte', 'Valor diárias', 'Valor passagens', 'Valor devolução', 'Valor outros gastos', 'Categoria Distância', 'Viagem Urgente', 'Justificativa Urgência Viagem', 'Distância (GCD)', 'Emissões (KgCO2eq)', 'Total_Emissoes_KgCO2eq', 'Total_Custo_R$', 'Teve_Trecho_Evitavel', 'Viagem_Urgente_Sem_Justificativa']


In [69]:
# --- CÉLULA DE DIAGNÓSTICO COMBINADA (Substitui as 3 células: Merge, Emissões e Limpeza) ---
import pandas as pd
import numpy as np

viagem_id_teste = '0000000000019375473'

def check_trip_exists(df, step_name):
    """Função auxiliar para verificar se a viagem de teste existe em um DataFrame."""
    print(f"\n--- Verificando após a etapa: {step_name} ---")
    print(f"Total de linhas no DataFrame: {len(df)}")
    if viagem_id_teste in df['Identificador do processo de viagem'].values:
        print(f"✅ SUCESSO: A viagem {viagem_id_teste} AINDA ESTÁ PRESENTE.")
    else:
        print(f"❌ FALHA: A viagem {viagem_id_teste} DESAPARECEU NESTA ETAPA!")
    print("-" * 50)

# --- INÍCIO DA LÓGICA DA CÉLULA DE MERGE ---
print(">>> INICIANDO ETAPA DE MERGE...")
df_master = trecho_df.copy()
check_trip_exists(df_master, "Cópia inicial de trecho_df")

df_master = pd.merge(df_master, passagem_df, on='Identificador do processo de viagem', how='left', suffixes=('', '_PASSAGEM'))
check_trip_exists(df_master, "Merge com passagem_df")

viagem_df_cleaned = viagem_df.drop(columns=['Número da Proposta (PCDP)', 'Meio de transporte'], errors='ignore')
viagem_df_cleaned = viagem_df_cleaned.drop_duplicates(subset=['Identificador do processo de viagem'], keep='first')
df_master = pd.merge(df_master, viagem_df_cleaned, on='Identificador do processo de viagem', how='left', suffixes=('', '_VIAGEM'))
check_trip_exists(df_master, "Merge com viagem_df")
print(">>> FIM DA ETAPA DE MERGE.")


# --- INÍCIO DA LÓGICA DA CÉLULA DE CÁLCULO DE EMISSÕES ---
print("\n>>> INICIANDO ETAPA DE CÁLCULO DE EMISSÕES...")
EF_MUITO_CURTA_EVITAVEL = 0.272576785
EF_CURTA_DISTANCIA = 0.182869354
EF_LONGA_DISTANCIA = 0.200108215
LIMITE_DH = 600
LIMITE_SH = 3700

conditions = [
    df_master['Distância (GCD)'].astype(float).le(LIMITE_DH),
    (df_master['Distância (GCD)'].astype(float).gt(LIMITE_DH)) & (df_master['Distância (GCD)'].astype(float).le(LIMITE_SH)),
    df_master['Distância (GCD)'].astype(float).gt(LIMITE_SH)
]
category_values = ['Muito Curta (Evitável)', 'Curta Distância', 'Longa Distância']
ef_values = [EF_MUITO_CURTA_EVITAVEL, EF_CURTA_DISTANCIA, EF_LONGA_DISTANCIA]

df_master['Categoria Distância'] = np.select(conditions, category_values, default='Não Calculada')
check_trip_exists(df_master, "Criação da 'Categoria Distância'")

df_master['EF Aplicado'] = np.select(conditions, ef_values, default=np.nan)
check_trip_exists(df_master, "Criação de 'EF Aplicado'")

df_master['Emissões (KgCO2eq)'] = df_master['Distância (GCD)'] * df_master['EF Aplicado']
check_trip_exists(df_master, "Cálculo de 'Emissões (KgCO2eq)'")
print(">>> FIM DA ETAPA DE CÁLCULO DE EMISSÕES.")


# --- INÍCIO DA LÓGICA DA CÉLULA DE LIMPEZA DE COLUNAS ---
print("\n>>> INICIANDO ETAPA DE LIMPEZA DE COLUNAS...")
# A lista de colunas a remover é baseada no seu notebook mais recente
colunas_para_remover = [
    'Sequência Trecho', 'Origem - Data', 'Destino - Data', 'Número Diárias', 'Missao?',
    'Origem - País', 'Origem - UF', 'Destino - País', 'Destino - UF',
    'Valor da passagem', 'Taxa de serviço', 'Data da emissão/compra', 'Hora da emissão/compra',
    'EF Aplicado'
]
colunas_redundantes_pos_merge = [col for col in df_master.columns if col.endswith('_PASSAGEM') or col.endswith('_VIAGEM')]
colunas_para_remover.extend(colunas_redundantes_pos_merge)

df_master = df_master.drop(columns=colunas_para_remover, errors='ignore')
check_trip_exists(df_master, "Remoção de colunas desnecessárias")
print(">>> FIM DA ETAPA DE LIMPEZA DE COLUNAS.")

>>> INICIANDO ETAPA DE MERGE...

--- Verificando após a etapa: Cópia inicial de trecho_df ---
Total de linhas no DataFrame: 1817646
✅ SUCESSO: A viagem 0000000000019375473 AINDA ESTÁ PRESENTE.
--------------------------------------------------

--- Verificando após a etapa: Merge com passagem_df ---
Total de linhas no DataFrame: 1817646
✅ SUCESSO: A viagem 0000000000019375473 AINDA ESTÁ PRESENTE.
--------------------------------------------------

--- Verificando após a etapa: Merge com viagem_df ---
Total de linhas no DataFrame: 1817646
✅ SUCESSO: A viagem 0000000000019375473 AINDA ESTÁ PRESENTE.
--------------------------------------------------
>>> FIM DA ETAPA DE MERGE.

>>> INICIANDO ETAPA DE CÁLCULO DE EMISSÕES...

--- Verificando após a etapa: Criação da 'Categoria Distância' ---
Total de linhas no DataFrame: 1817646
✅ SUCESSO: A viagem 0000000000019375473 AINDA ESTÁ PRESENTE.
--------------------------------------------------

--- Verificando após a etapa: Criação de 'EF Aplica

In [74]:
# --- AGREGAÇÃO DAS VIAGENS NO DF_MASTER (VERSÃO FINAL COMPLETA) ---
print("🔄 Agregando distâncias no df_master por Identificador do processo de viagem...")

df_agrupado = (
    df_master.groupby("Identificador do processo de viagem", as_index=False)
    .agg({
        # --- Principais campos de viagem ---
        "Identificador do processo de viagem": "first",
        "Número da Proposta (PCDP)": "first",
        "Cod Trecho Temp": "first",
        "Situação": "first",

        # --- Campos institucionais ---
        "Código do órgão superior": "first",
        "Nome do órgão superior": "first",
        "Código órgão solicitante": "first",
        "Nome órgão solicitante": "first",

        # --- Dados pessoais e funcionais ---
        "CPF viajante": "first",
        "Nome": "first",
        "Cargo": "first",
        "Função": "first",
        "Descrição Função": "first",

        # --- Informações da viagem ---
        "Destinos": "first",
        "Motivo": "first",
        "Origem - Cidade": "first",
        "Destino - Cidade": "last",
        "Meio de transporte": "first",
        "Situação": "first",

        #

        # --- Custos e valores ---
        "Valor diárias": "first",
        "Valor passagens": "first",
        "Valor devolução": "first",
        "Valor outros gastos": "first",


        # --- Indicadores e flags ---
        "Categoria Distância": "first",
        "Viagem Urgente": "first",
        "Justificativa Urgência Viagem": "first",
        
        #  --- Cálculos principais ---
        "Distância (GCD)": "sum",
        "Emissões (KgCO2eq)": "sum",
    })
)

df_master = df_agrupado.copy()
print(f"✅ df_master agregado: {len(df_master)} viagens únicas (com todas as colunas relevantes).")


🔄 Agregando distâncias no df_master por Identificador do processo de viagem...
✅ df_master agregado: 714013 viagens únicas (com todas as colunas relevantes).


In [75]:
df_master.head()

,Identificador do processo de viagem,Número da Proposta (PCDP),Cod Trecho Temp,Situação,Código do órgão superior,Nome do órgão superior,Código órgão solicitante,Nome órgão solicitante,CPF viajante,Nome,...,Meio de transporte,Valor diárias,Valor passagens,Valor devolução,Valor outros gastos,Categoria Distância,Viagem Urgente,Justificativa Urgência Viagem,Distância (GCD),Emissões (KgCO2eq)
0,0000000000017821923,000001/23-1C,São Paulo_Loughborough,Realizada,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,PRISCILA LEAL DA SILVA,...,Aéreo,"0,00","0,00","0,00","0,00",Longa Distância,NÃO,Sem informação,19105.768888,3823.221308
1,0000000000018159396,000001/23,Rio de Janeiro_Pirenópolis,Realizada,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,Ekaterina Pavlovskaia,...,Rodoviário,"0,00","7894,50","0,00","0,00",Curta Distância,NÃO,Sem informação,21364.183042,4241.036516
2,0000000000018236583,000018/23,Rio de Janeiro_Niterói,Realizada,26000,Ministério da Educação,26236,Universidade Federal Fluminense,None,CAMILLA DUARTE DA SILVA,...,Veículo Próprio,"0,00","0,00","0,00","0,00",Muito Curta (Evitável),NÃO,Sem informação,21.032762,5.733043
3,0000000000018288418,000007/23-1C,Nova York_Brasília,Realizada,52000,Ministério da Defesa,52121,Comando do Exército,***.621.358-**,ANDRE LUIS COSTA PITANGUEIRA,...,Aéreo,"39565,70","7434,13","0,00","1095,35",Longa Distância,SIM,Por necessidade do serviço.,13667.785393,2735.036138
4,0000000000018296348,000070/23,Vitória_Alegre,Realizada,26000,Ministério da Educação,26406,"Instituto Federal de Educação, Ciência e Tecno...",***.407.547-**,LIDIANY MIRANDA FERRAZ NUNES,...,Rodoviário,"1013,96","0,00","0,00","0,00",Muito Curta (Evitável),SIM,A efetivação do empenho no Siafi e a respectiv...,273.418829,74.527625


# Agregação

In [26]:
# --- CÓDIGO DE DIAGNÓSTICO (INSERIR NO TOPO DA CÉLULA DE AGREGAÇÃO) ---
viagem_id_teste = '0000000000019375473' # ID da viagem que deu metade da emissão

print(f"--- Verificando o df_master para a viagem ID {viagem_id_teste} ANTES da agregação ---")

# Filtra o df_master para o ID específico e mostra as colunas relevantes
df_inspecao = df_master[df_master['Identificador do processo de viagem'] == viagem_id_teste]

if df_inspecao.empty:
    print(f"❌ ERRO CRÍTICO: A viagem {viagem_id_teste} não foi encontrada no df_master.")
else:
    print(df_inspecao[['Origem - Cidade', 'Destino - Cidade', 'Distância (GCD)', 'Emissões (KgCO2eq)']])
    print(f"\n>>> Número de trechos encontrados para esta viagem: {len(df_inspecao)}\n")
    if len(df_inspecao) < 2:
        print("✅ DIAGNÓSTICO CONFIRMADO: Há apenas 1 trecho no DataFrame. É por isso que a soma das emissões está resultando na metade do esperado. O problema está no MERGE da célula anterior.")
    else:
        print("⚠️ AVISO: Os dois trechos (ida e volta) estão presentes. A causa do erro é outra.")
# --- FIM DO CÓDIGO DE DIAGNÓSTICO ---

--- Verificando o df_master para a viagem ID 0000000000019375473 ANTES da agregação ---
        Origem - Cidade Destino - Cidade  Distância (GCD)  Emissões (KgCO2eq)
1325147     João Pessoa         Valencia      6277.191835         1256.117653
1325148        Valencia      João Pessoa      6277.191835         1256.117653

>>> Número de trechos encontrados para esta viagem: 2

⚠️ AVISO: Os dois trechos (ida e volta) estão presentes. A causa do erro é outra.


# Agregar

In [76]:
# --- CÉLULA DE AGREGAÇÃO ---
import numpy as np

# 1. Conversão de Tipos para Colunas de Custo
COLUNAS_CUSTO = ['Valor diárias', 'Valor outros gastos', 'Valor passagens']
for col in COLUNAS_CUSTO:
    df_master[col] = pd.to_numeric(df_master[col].astype(str).str.replace(',', '.', regex=True), errors='coerce')

# 2. Cálculo dos Totais usando .transform()
#    .transform('sum') calcula a soma por grupo e alinha o resultado de volta ao df_master original.
#    Isso corrige o erro que calculava a emissão pela metade.
df_master['Total_Emissoes_KgCO2eq'] = df_master.groupby('Identificador do processo de viagem')['Emissões (KgCO2eq)'].transform('sum')

# Para o custo, como os valores da viagem toda se repetem, usamos .transform('first') para pegar o primeiro valor de cada grupo
df_master['Total_Custo_R$'] = (
    df_master.groupby('Identificador do processo de viagem')['Valor passagens'].transform('first').fillna(0) +
    df_master.groupby('Identificador do processo de viagem')['Valor diárias'].transform('first').fillna(0) +
    df_master.groupby('Identificador do processo de viagem')['Valor outros gastos'].transform('first').fillna(0)
)

# 3. Cálculo dos Flags (indicadores booleanos)
#    Usa .transform() para garantir que o resultado seja aplicado a todas as linhas da mesma viagem
df_master['Teve_Trecho_Evitavel'] = df_master.groupby('Identificador do processo de viagem')['Categoria Distância'].transform(
    lambda x: ('Muito Curta (Evitável)' in x.values)
)

URGENTE_SIM = 'SIM'
SEM_INFO = 'Sem informação'
df_master['Viagem_Urgente_Sem_Justificativa'] = (
    (df_master['Viagem Urgente'].astype(str).str.upper().str.strip() == URGENTE_SIM) &
    (df_master['Justificativa Urgência Viagem'].isna() | (df_master['Justificativa Urgência Viagem'].astype(str).str.upper().str.strip() == SEM_INFO))
)

print("✅ Colunas de totais ('Total_Emissoes_KgCO2eq', 'Total_Custo_R$') e flags calculadas com '.transform()'.")

✅ Colunas de totais ('Total_Emissoes_KgCO2eq', 'Total_Custo_R$') e flags calculadas com '.transform()'.


In [77]:
df_master.head()

,Identificador do processo de viagem,Número da Proposta (PCDP),Cod Trecho Temp,Situação,Código do órgão superior,Nome do órgão superior,Código órgão solicitante,Nome órgão solicitante,CPF viajante,Nome,...,Valor outros gastos,Categoria Distância,Viagem Urgente,Justificativa Urgência Viagem,Distância (GCD),Emissões (KgCO2eq),Total_Emissoes_KgCO2eq,Total_Custo_R$,Teve_Trecho_Evitavel,Viagem_Urgente_Sem_Justificativa
0,0000000000017821923,000001/23-1C,São Paulo_Loughborough,Realizada,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,PRISCILA LEAL DA SILVA,...,0.00,Longa Distância,NÃO,Sem informação,19105.768888,3823.221308,3823.221308,0.00,False,False
1,0000000000018159396,000001/23,Rio de Janeiro_Pirenópolis,Realizada,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.000.000-**,Ekaterina Pavlovskaia,...,0.00,Curta Distância,NÃO,Sem informação,21364.183042,4241.036516,4241.036516,7894.50,False,False
2,0000000000018236583,000018/23,Rio de Janeiro_Niterói,Realizada,26000,Ministério da Educação,26236,Universidade Federal Fluminense,None,CAMILLA DUARTE DA SILVA,...,0.00,Muito Curta (Evitável),NÃO,Sem informação,21.032762,5.733043,5.733043,0.00,True,False
3,0000000000018288418,000007/23-1C,Nova York_Brasília,Realizada,52000,Ministério da Defesa,52121,Comando do Exército,***.621.358-**,ANDRE LUIS COSTA PITANGUEIRA,...,1095.35,Longa Distância,SIM,Por necessidade do serviço.,13667.785393,2735.036138,2735.036138,48095.18,False,False
4,0000000000018296348,000070/23,Vitória_Alegre,Realizada,26000,Ministério da Educação,26406,"Instituto Federal de Educação, Ciência e Tecno...",***.407.547-**,LIDIANY MIRANDA FERRAZ NUNES,...,0.00,Muito Curta (Evitável),SIM,A efetivação do empenho no Siafi e a respectiv...,273.418829,74.527625,74.527625,1013.96,True,False


In [29]:
df_master.columns

Index(['Identificador do processo de viagem', 'Número da Proposta (PCDP)',
       'Origem - Cidade', 'Destino - Cidade', 'Meio de transporte',
       'Cod Trecho Temp', 'Distância (GCD)', 'Situação', 'Viagem Urgente',
       'Justificativa Urgência Viagem', 'Código do órgão superior',
       'Nome do órgão superior', 'Código órgão solicitante',
       'Nome órgão solicitante', 'CPF viajante', 'Nome', 'Cargo', 'Função',
       'Descrição Função', 'Período - Data de início', 'Período - Data de fim',
       'Destinos', 'Motivo', 'Valor diárias', 'Valor passagens',
       'Valor devolução', 'Valor outros gastos', 'País - Origem ida',
       'UF - Origem ida', 'Cidade - Origem ida', 'País - Destino ida',
       'UF - Destino ida', 'Cidade - Destino ida', 'País - Origem volta',
       'UF - Origem volta', 'Cidade - Origem volta', 'Pais - Destino volta',
       'UF - Destino volta', 'Cidade - Destino volta', 'Categoria Distância',
       'Emissões (KgCO2eq)', 'Total_Emissoes_KgCO2eq', 'Total_

In [78]:
# --- DIAGNÓSTICO DO CÁLCULO DE DISTÂNCIA E MERGE ---

# 1. Sucesso no cálculo de Distância (Resultado da Célula 3)
# df_trechos_calculados é o DataFrame que contém os trechos únicos e a Distância (GCD)
total_unicos = len(df_trechos_calculados)
sucesso_calculo = df_trechos_calculados['Distância (GCD)'].notna().sum()
perc_sucesso_calculo = (sucesso_calculo / total_unicos) * 100

print("--- DIAGNÓSTICO DO CÁLCULO DE DISTÂNCIA ---")
print(f"Total de Trechos Únicos Tentados: {total_unicos}")
print(f"Trechos com Distância Calculada: {sucesso_calculo}")
print(f"Taxa de Sucesso no Cálculo (API): {perc_sucesso_calculo:.2f}%")

# 2. Sucesso no Merge (Resultado no DataFrame Final)
total_master = len(df_master)
sucesso_merge = df_master['Distância (GCD)'].notna().sum()
perc_sucesso_merge = (sucesso_merge / total_master) * 100

print("\n--- DIAGNÓSTICO NO DATAFRAME FINAL (df_master) ---")
print(f"Total de Linhas em df_master: {total_master}")
print(f"Linhas em df_master com Distância: {sucesso_merge}")
print(f"Percentual de Linhas com Distância: {perc_sucesso_merge:.2f}%")

if sucesso_merge == 0:
    print("\n⚠️ AVISO CRÍTICO: Zero distâncias foram calculadas e/ou merge falhou. A causa é provavelmente o bloqueio da API.")

--- DIAGNÓSTICO DO CÁLCULO DE DISTÂNCIA ---
Total de Trechos Únicos Tentados: 100211
Trechos com Distância Calculada: 100211
Taxa de Sucesso no Cálculo (API): 100.00%

--- DIAGNÓSTICO NO DATAFRAME FINAL (df_master) ---
Total de Linhas em df_master: 714013
Linhas em df_master com Distância: 714013
Percentual de Linhas com Distância: 100.00%


In [ ]:
# # --- CÉLULA NOVA: CONVERTER COLUNAS DE CUSTO PARA INTEIRO ---

# # Lista das colunas de custo que devem ser convertidas
# colunas_custo_para_int = [
#     'Valor diárias',
#     'Valor passagens',
#     'Valor outros gastos',
#     'Total_Custo_R$',
#     'Emissões (KgCO2eq)'
# ]

# print("Convertendo colunas de custo para o tipo inteiro...")

# for col in colunas_custo_para_int:
#     # Verifica se a coluna realmente existe no DataFrame antes de tentar a conversão
#     if col in df_master.columns:
#         # 1. Preenche valores ausentes (NaN), que não podem ser convertidos para int, com 0.
#         df_master[col] = df_master[col].fillna(0)

#         # 2. Converte a coluna para o tipo inteiro padrão.
#         df_master[col] = df_master[col].astype(int)
#         print(f" - Coluna '{col}' foi convertida para inteiro.")
#     else:
#         print(f" - Aviso: A coluna '{col}' não foi encontrada para conversão.")

# print("\n✅ Conversão concluída. O DataFrame 'df_master' está pronto para ser salvo.")

Convertendo colunas de custo para o tipo inteiro...
 - Coluna 'Valor diárias' foi convertida para inteiro.
 - Coluna 'Valor passagens' foi convertida para inteiro.
 - Coluna 'Valor outros gastos' foi convertida para inteiro.
 - Coluna 'Total_Custo_R$' foi convertida para inteiro.
 - Coluna 'Emissões (KgCO2eq)' foi convertida para inteiro.

✅ Conversão concluída. O DataFrame 'df_master' está pronto para ser salvo.


In [80]:
# --- CÉLULA DE LIMPEZA DO DF_MASTER ---

print("Colunas originais:", len(df_master.columns))

# Lista das colunas que são redundantes ou desnecessárias para a análise final
colunas_para_remover = [
    # Coluna duplicada do 'Número da Proposta'
    'Número da Proposta (PCDP)_TRECHO',

    # Colunas de Origem/Destino do arquivo Viagem.csv (redundantes)
    'País - Origem ida',
    'UF - Origem ida',
    'Cidade - Origem ida',
    'País - Destino ida',
    'UF - Destino ida',
    'Cidade - Destino ida',
    'País - Origem volta',
    'UF - Origem volta',
    'Cidade - Origem volta',
    'Pais - Destino volta', 
    'UF - Destino volta',
    'Cidade - Destino volta'
]

# Remove as colunas da lista, ignorando erros caso alguma já tenha sido removida
df_master = df_master.drop(columns=colunas_para_remover, errors='ignore')

print("✅ Colunas desnecessárias removidas com sucesso.")
print("Colunas restantes:", len(df_master.columns))
print("\nLista de colunas final no df_master:")
print(df_master.columns)

Colunas originais: 31
✅ Colunas desnecessárias removidas com sucesso.
Colunas restantes: 31

Lista de colunas final no df_master:
Index(['Identificador do processo de viagem', 'Número da Proposta (PCDP)',
       'Cod Trecho Temp', 'Situação', 'Código do órgão superior',
       'Nome do órgão superior', 'Código órgão solicitante',
       'Nome órgão solicitante', 'CPF viajante', 'Nome', 'Cargo', 'Função',
       'Descrição Função', 'Destinos', 'Motivo', 'Origem - Cidade',
       'Destino - Cidade', 'Meio de transporte', 'Valor diárias',
       'Valor passagens', 'Valor devolução', 'Valor outros gastos',
       'Categoria Distância', 'Viagem Urgente',
       'Justificativa Urgência Viagem', 'Distância (GCD)',
       'Emissões (KgCO2eq)', 'Total_Emissoes_KgCO2eq', 'Total_Custo_R$',
       'Teve_Trecho_Evitavel', 'Viagem_Urgente_Sem_Justificativa'],
      dtype='object')


In [81]:
import os
local = f'dadosViagens/dados_viagens{ano}'
if not os.path.exists(local):
    print(f"📁 Diretório '{local}' não encontrado.")

df_master.to_csv(f'{local}/df_master_viagens_a_servico_{ano}.csv', index=False)
print(f"\n✅ DataFrame Mestre salvo como '{local}/df_master_viagens_a_servico_{ano}.csv'.")


✅ DataFrame Mestre salvo como 'dadosViagens/dados_viagens2023/df_master_viagens_a_servico_2023.csv'.


In [35]:
# carregar o df_master salvo
import pandas as pd
ano = 2023
local = f'dadosViagens/dados_viagens{ano}'
df_master = pd.read_csv(f'{local}/df_master_viagens_a_servico_{ano}.csv')
print(f"✅ DataFrame Mestre carregado de '{local}/df_master_viagens_a_servico_{ano}.csv' com {len(df_master)} linhas.")

✅ DataFrame Mestre carregado de 'dadosViagens/dados_viagens2023/df_master_viagens_a_servico_2023.csv' com 1817646 linhas.


In [82]:
len(df_master.columns)

31

In [83]:
df_master.columns

Index(['Identificador do processo de viagem', 'Número da Proposta (PCDP)',
       'Cod Trecho Temp', 'Situação', 'Código do órgão superior',
       'Nome do órgão superior', 'Código órgão solicitante',
       'Nome órgão solicitante', 'CPF viajante', 'Nome', 'Cargo', 'Função',
       'Descrição Função', 'Destinos', 'Motivo', 'Origem - Cidade',
       'Destino - Cidade', 'Meio de transporte', 'Valor diárias',
       'Valor passagens', 'Valor devolução', 'Valor outros gastos',
       'Categoria Distância', 'Viagem Urgente',
       'Justificativa Urgência Viagem', 'Distância (GCD)',
       'Emissões (KgCO2eq)', 'Total_Emissoes_KgCO2eq', 'Total_Custo_R$',
       'Teve_Trecho_Evitavel', 'Viagem_Urgente_Sem_Justificativa'],
      dtype='object')

In [84]:
#filtra para viagens aereas mantendo todas as colunas

#O filtro deve ser feito no df_master e usar .copy()
aereas = df_master[df_master['Meio de transporte'].astype(str).str.upper().str.strip() == 'AÉREO'].copy()

In [85]:
len(aereas.columns)

31

In [86]:
aereas.head()

,Identificador do processo de viagem,Número da Proposta (PCDP),Cod Trecho Temp,Situação,Código do órgão superior,Nome do órgão superior,Código órgão solicitante,Nome órgão solicitante,CPF viajante,Nome,...,Valor outros gastos,Categoria Distância,Viagem Urgente,Justificativa Urgência Viagem,Distância (GCD),Emissões (KgCO2eq),Total_Emissoes_KgCO2eq,Total_Custo_R$,Teve_Trecho_Evitavel,Viagem_Urgente_Sem_Justificativa
0,0000000000017821923,000001/23-1C,São Paulo_Loughborough,Realizada,26000,Ministério da Educação,26352,Fundação Universidade Federal do ABC,***.875.238-**,PRISCILA LEAL DA SILVA,...,0.00,Longa Distância,NÃO,Sem informação,19105.768888,3823.221308,3823.221308,0.00,False,False
3,0000000000018288418,000007/23-1C,Nova York_Brasília,Realizada,52000,Ministério da Defesa,52121,Comando do Exército,***.621.358-**,ANDRE LUIS COSTA PITANGUEIRA,...,1095.35,Longa Distância,SIM,Por necessidade do serviço.,13667.785393,2735.036138,2735.036138,48095.18,False,False
5,0000000000018302983,000001/23,Gotemburgo_São Paulo,Realizada,52000,Ministério da Defesa,52111,Comando da Aeronáutica,***.855.388-**,GREGOR GASPAR,...,0.00,Longa Distância,NÃO,Sem informação,21066.098073,4215.499282,4215.499282,14903.10,False,False
6,0000000000018306758,000002/23,Gotemburgo_São Paulo,Realizada,52000,Ministério da Defesa,52111,Comando da Aeronáutica,***.295.391-**,VITOR LUIS MARTINS FARIA,...,0.00,Longa Distância,NÃO,Sem informação,21066.098073,4215.499282,4215.499282,14903.10,False,False
7,0000000000018306785,000004/23,Gotemburgo_São Paulo,Realizada,52000,Ministério da Defesa,52111,Comando da Aeronáutica,***.221.068-**,RAFAEL RODRIGO MANCIN DE MORAIS,...,0.00,Longa Distância,NÃO,Sem informação,21066.098073,4215.499282,4215.499282,14903.10,False,False


In [87]:
import os
# salvar aereas 
local = f'dadosViagens/dados_viagens{ano}'
if not os.path.exists(local):
    print(f"📁 Diretório '{local}' não encontrado.")
    
aereas.to_csv(f'{local}/aereas_viagens_a_servico_{ano}.csv', index=False)
print(f"\n✅ DataFrame de Viagens Aéreas salvo como 'aereas")


✅ DataFrame de Viagens Aéreas salvo como 'aereas


In [42]:
# carregar o csv de viagens aereas
import pandas as pd
ano = 2023
local = f'dadosViagens/dados_viagens{ano}'
aereas = pd.read_csv(f'{local}/aereas_viagens_a_servico_{ano}.csv')

In [43]:
aereas.head()

,Identificador do processo de viagem,Número da Proposta (PCDP),Origem - Cidade,Destino - Cidade,Meio de transporte,Cod Trecho Temp,Distância (GCD),Situação,Viagem Urgente,Justificativa Urgência Viagem,...,Valor diárias,Valor passagens,Valor devolução,Valor outros gastos,Categoria Distância,Emissões (KgCO2eq),Total_Emissoes_KgCO2eq,Total_Custo_R$,Teve_Trecho_Evitavel,Viagem_Urgente_Sem_Justificativa
0,17821923,000001/23-1C,São Paulo,Loughborough,Aéreo,São Paulo_Loughborough,9552.884444,Realizada,NÃO,Sem informação,...,0,0,"0,00",0,Longa Distância,1911,3823.221308,0,False,False
1,17821923,000001/23-1C,Loughborough,São Paulo,Aéreo,Loughborough_São Paulo,9552.884444,Realizada,NÃO,Sem informação,...,0,0,"0,00",0,Longa Distância,1911,3823.221308,0,False,False
2,18159396,000001/23,Rio de Janeiro,Aberdeen,Aéreo,Rio de Janeiro_Aberdeen,9692.698490,Realizada,NÃO,Sem informação,...,0,7894,"0,00",0,Longa Distância,1939,4241.036516,7894,False,False
3,18159396,000001/23,Aberdeen,Rio de Janeiro,Aéreo,Aberdeen_Rio de Janeiro,9692.698490,Realizada,NÃO,Sem informação,...,0,7894,"0,00",0,Longa Distância,1939,4241.036516,7894,False,False
4,18288418,000007/23-1C,Nova York,Brasília,Aéreo,Nova York_Brasília,6833.892697,Realizada,SIM,Por necessidade do serviço.,...,39565,7434,"0,00",1095,Longa Distância,1367,2735.036138,48095,False,False


In [88]:
import numpy as np
import pandas as pd

COLUNA_DISTANCIA = 'Distância (GCD)'
LIMITE_MAXIMO_KM = 21000 # Limite físico para detecção de erro numérico

# 1. Contagem de NaN (Viagens que não tiveram distância calculada)
nan_count = aereas[COLUNA_DISTANCIA].isna().sum()

# 2. Contagem de Distâncias Absurdas (> 21.000 km)
absurd_count = (aereas[COLUNA_DISTANCIA] > LIMITE_MAXIMO_KM).sum()

# 3. Métricas
max_distance = aereas[COLUNA_DISTANCIA].max()
mean_distance = aereas[aereas[COLUNA_DISTANCIA].notna() & (aereas[COLUNA_DISTANCIA] > 0)][COLUNA_DISTANCIA].mean()


print("--- Diagnóstico Final de Distância (GCD) em 'aereas' ---")
print(f"1. Valor MÁXIMO Encontrado: {max_distance:,.2f} km")
print(f"2. Número de Viagens com Distância Impossível (> {LIMITE_MAXIMO_KM} km): {absurd_count}")
print(f"3. Número de Viagens com Distância Faltando (NaN): {nan_count}")
print(f"4. Média de Distância por Viagem Válida: {mean_distance:,.2f} km")

if absurd_count > 0:
    print("\n🚨 CRÍTICO: O filtro pós-processamento falhou. Há distâncias > 21.000 km.")
    print("Sugestão: Verifique o log da Célula 3 para garantir que o filtro foi aplicado.")

--- Diagnóstico Final de Distância (GCD) em 'aereas' ---
1. Valor MÁXIMO Encontrado: 85,653.57 km
2. Número de Viagens com Distância Impossível (> 21000 km): 3036
3. Número de Viagens com Distância Faltando (NaN): 0
4. Média de Distância por Viagem Válida: 3,754.25 km

🚨 CRÍTICO: O filtro pós-processamento falhou. Há distâncias > 21.000 km.
Sugestão: Verifique o log da Célula 3 para garantir que o filtro foi aplicado.


# viagens sem distancia 

In [45]:
import pandas as pd

# Filtro para encontrar ocorrências com Distância NaN
# O método .isna() retorna True para todos os valores NaN
df_viagens_sem_distancia = aereas[aereas['Distância (GCD)'].isna()].copy()

print(f"✅ Total de viagens sem Distância calculada (NaN): {len(df_viagens_sem_distancia)}")

# Para inspecionar as viagens que falharam, exiba o resultado:
print("\n--- Amostra das viagens que falharam (NaN) ---")
# Mostra o ID da viagem e os custos (para ter contexto)
print(df_viagens_sem_distancia[['Identificador do processo de viagem', 'Total_Custo_R$']].head())

✅ Total de viagens sem Distância calculada (NaN): 0

--- Amostra das viagens que falharam (NaN) ---
Empty DataFrame
Columns: [Identificador do processo de viagem, Total_Custo_R$]
Index: []


In [46]:
# --- CÉLULA DE DIAGNÓSTICO DE DADOS FALTANTES (Com Colunas Corretas) ---

# Colunas a serem verificadas
COLUNAS_GEOGRAFICAS = ['Destinos', 'Origem - Cidade', 'Destino - Cidade'] # Usando nomes das colunas agregadas

# 1. Criação da Máscara de Valores Faltantes (NaN ou string vazia)
# Vamos verificar se ALGUMA das colunas está faltando.
mask_faltante = pd.DataFrame()

# Cria a máscara verificando NaN ou strings vazias em cada coluna
for col in COLUNAS_GEOGRAFICAS:
    mask = (aereas[col].isna()) | (aereas[col].astype(str).str.strip() == '')
    if mask_faltante.empty:
        mask_faltante = mask
    else:
        mask_faltante = mask_faltante | mask # Use OR lógico para capturar qualquer falha

# 2. Filtrar e exibir as viagens com dados faltantes
df_viagens_faltantes = aereas[mask_faltante].copy()


print("--- Viagens com Dados Geográficos Faltantes ---")
print(f"Total de viagens únicas no 'aereas': {len(aereas)}")
print(f"Total de viagens únicas com dados de cidade/destino faltantes: {len(df_viagens_faltantes)}")

# Exibe as colunas relevantes para diagnóstico
df_viagens_faltantes[['Identificador do processo de viagem', 'Destinos', 'Origem - Cidade', 'Destino - Cidade']].head()

--- Viagens com Dados Geográficos Faltantes ---
Total de viagens únicas no 'aereas': 544928
Total de viagens únicas com dados de cidade/destino faltantes: 16621


,Identificador do processo de viagem,Destinos,Origem - Cidade,Destino - Cidade
23,18357399,NaN,Rio de Janeiro,Brasília
33,18387433,NaN,Belém,Brasília
34,18387433,NaN,Brasília,Belém
134,18462146,NaN,Salvador,Belo Horizonte
135,18462146,NaN,Belo Horizonte,Salvador


In [47]:
# # apagar os registros com dados faltantes que impossibilitam o cálculo de distância que estar na variável df_viagens_faltantes
# print(f"\nIniciando remoção de {len(df_viagens_faltantes)} registros com dados faltantes...")
# aereas_limpo = aereas[~aereas['Identificador do processo de viagem'].isin(df_viagens_faltantes['Identificador do processo de viagem'])].copy()
# print(f"✅ Remoção concluída. Registros restantes em 'aereas_limpo': {len(aereas_limpo)}")


# Filtrando para UFPB

In [48]:
# --- CÉLULA DE FILTRAGEM UFPB (FINAL) ---

# Tenta cobrir as duas variações mais comuns (com acento e sem acento + sigla)
FILTRO_UFPB_ROBUSTO = 'UFPB|UNIVERSIDADE FEDERAL DA PARAIBA|UNIVERSIDADE FEDERAL DA PARAÍBA'

aereas_ufpb = aereas[
    aereas['Nome órgão solicitante'].astype(str).str.upper().str.contains(FILTRO_UFPB_ROBUSTO, na=False)
].copy()


if len(aereas_ufpb) > 0:
    print(f"✅ Filtragem UFPB concluída com sucesso. Total de viagens encontradas: {len(aereas_ufpb)}")
    
else:
    # --- DIAGNÓSTICO FINAL DE CONTEÚDO (Se a filtragem falhou) ---
    print("❌ FALHA CRÍTICA: Nenhum filtro funcionou. O nome do órgão é diferente.")
    
    # Tentativa de encontrar e imprimir a string exata do DataFrame para que o usuário copie.
    if 'Nome órgão solicitante' in aereas.columns and not aereas.empty:
        try:
            # Tenta encontrar a primeira ocorrência que contenha 'Paraíba' ou 'Paraiba'
            nome_real = aereas[
                aereas['Nome órgão solicitante'].astype(str).str.contains('Paraíba|Paraiba', na=False)
            ].iloc[0]['Nome órgão solicitante']
            
            print(f"⚠️ VALOR LIDO: Copie o nome exato para usar no filtro: '{nome_real}'")
            print(f"   Use o filtro: aereas_ufpb = aereas[aereas['Nome órgão solicitante'] == '{nome_real}'].copy()")
            
        except IndexError:
            print("Não foi possível encontrar 'Paraíba' na coluna. O nome do órgão é totalmente diferente ou o DF está vazio.")

✅ Filtragem UFPB concluída com sucesso. Total de viagens encontradas: 1287


In [49]:
# salvar aereas_ufpb
import os
local = f'dadosViagens/dados_viagens{ano}'
if not os.path.exists(local):
    print(f"📁 Diretório '{local}' não encontrado.")
aereas_ufpb.to_csv(f'{local}/aereas_ufpb_viagens_a_servico_{ano}.csv', index=False)
print(f"\n✅ DataFrame de Viagens Aéreas da UFPB salvo como '{local}/aereas_ufpb_viagens_a_servico_{ano}.csv'.")



✅ DataFrame de Viagens Aéreas da UFPB salvo como 'dadosViagens/dados_viagens2023/aereas_ufpb_viagens_a_servico_2023.csv'.


In [50]:
#carregar o csv de viagens aereas da ufpb
import pandas as pd 
ano = 2023
local = f'dadosViagens/dados_viagens{ano}'
aereas_ufpb = pd.read_csv(f'{local}/aereas_ufpb_viagens_a_servico_{ano}.csv')

In [51]:
# ver os  "indentificador do processo de viagem" únicos
print(len( aereas_ufpb))
indentificador = aereas_ufpb["Identificador do processo de viagem"].unique()
print(len(indentificador))

1287
637


In [53]:
# --- CÉLULA REVISADA FINAL: OBTER DATAFRAME DE VIAGENS ÚNICAS ---
import numpy as np
import pandas as pd

# 1. Calcular a Distância Total Correta da Viagem
#    Isso é necessário porque 'Distância (GCD)' em 'aereas_ufpb' é por trecho.
#    Somamos a distância de cada TRECHO ÚNICO para obter a distância total da VIAGEM.
distancia_correta_por_viagem = aereas_ufpb.drop_duplicates(
    subset=['Identificador do processo de viagem', 'Origem - Cidade', 'Destino - Cidade', 'Distância (GCD)']
).groupby('Identificador do processo de viagem')['Distância (GCD)'].sum().reset_index()


# 2. Obter DataFrame de Viagens ÚNICAS
#    NÃO vamos usar 'groupby().agg()' aqui para evitar re-somar os custos.
#    Em vez disso, vamos simplesmente remover as linhas duplicadas por viagem.
#    A primeira linha de cada viagem já contém os TOTAIS corretos (exceto a distância).
df_viagens_unificadas = aereas_ufpb.drop_duplicates(
    subset=['Identificador do processo de viagem'],
    keep='first'
).copy()


# 3. Atualizar a Distância no DataFrame de Viagens Únicas
#    Primeiro, removemos a coluna de distância do trecho individual.
df_viagens_unificadas = df_viagens_unificadas.drop(columns=['Distância (GCD)'])

#    Agora, fazemos o merge para adicionar a coluna com a SOMA CORRETA da distância.
df_viagens_unificadas = pd.merge(
    df_viagens_unificadas,
    distancia_correta_por_viagem, # Este DF tem o ID da viagem e a soma correta da distância
    on="Identificador do processo de viagem",
    how="left"
)


# 4. Ordenar e Converter Tipos (Opcional, mas recomendado para limpeza)
df_viagens_unificadas = df_viagens_unificadas.sort_values(
    by="Distância (GCD)",
    ascending=False
)
df_viagens_unificadas['Total_Emissoes_KgCO2eq'] = df_viagens_unificadas['Total_Emissoes_KgCO2eq'].round().astype('Int64')
df_viagens_unificadas['Distância (GCD)'] = df_viagens_unificadas['Distância (GCD)'].fillna(0).round().astype('Int64')
print("✅ Colunas numéricas agregadas por viagem arredondadas e convertidas para inteiro.")


# 5. Salvar em Dicionário
viagens_dict = df_viagens_unificadas.set_index("Identificador do processo de viagem").to_dict(orient='index')

print("✅ Obtenção do DataFrame de Viagens Únicas ('df_viagens_unificadas') concluída.")
print(f"Total de Viagens Únicas: {len(df_viagens_unificadas)}")
print(f"Dicionário 'viagens_dict' criado com {len(viagens_dict)} entradas.")

print("\n--- Amostra das 5 Viagens com Maior Distância (DF Unificado FINAL) ---")
# Exibe colunas chave para verificar. Os valores agora devem estar corretos.
print(df_viagens_unificadas[[
    'Identificador do processo de viagem',
    'Total_Emissoes_KgCO2eq',
    'Distância (GCD)',
    'Total_Custo_R$'
]].head())

✅ Colunas numéricas agregadas por viagem arredondadas e convertidas para inteiro.
✅ Obtenção do DataFrame de Viagens Únicas ('df_viagens_unificadas') concluída.
Total de Viagens Únicas: 637
Dicionário 'viagens_dict' criado com 637 entradas.

--- Amostra das 5 Viagens com Maior Distância (DF Unificado FINAL) ---
     Identificador do processo de viagem  Total_Emissoes_KgCO2eq  \
634                             19595605                    8014   
0                               18491955                    6720   
99                              18954786                    6657   
612                             19546429                    6156   
172                             19082648                    6428   

     Distância (GCD)  Total_Custo_R$  
634            40427            8869  
0              33581           24008  
99             31759           16691  
612            30763           11151  
172            30763           11090  


In [54]:
# distancia = aereas_ufpb["Distância (GCD)"].sort_values()

# distancia.head(10)
# a = aereas_ufpb[aereas_ufpb["Distância (GCD)"] == 2262655]
# a

zeros =df_viagens_unificadas["Distância (GCD)"]
# ver distancia com valores igual a zero 
zeros = zeros[zeros == 0]
print(f"Total de viagens com distância igual a zero: {len(zeros)}")
# pegar_zeros = df_viagens_unificadas[df_viagens_unificadas["Distância (GCD)"] == 0]
# pegar_zeros
# # # apagar as linhas com distancia igual a zero
# aereas_ufpb = aereas_ufpb[aereas_ufpb["Distância (GCD)"] != 0]
# print(f"Total de viagens após remover distâncias iguais a zero: {len(aereas_ufpb)}")

Total de viagens com distância igual a zero: 0


In [55]:
print(df_viagens_unificadas.head(10)[['Distância (GCD)']])

     Distância (GCD)
634            40427
0              33581
99             31759
612            30763
172            30763
284            20459
1              19222
20             18421
14             18421
371            13338


In [56]:
distancia  = df_viagens_unificadas['Distância (GCD)'].sort_values(ascending=False)
print(distancia)

634    40427
0      33581
99     31759
612    30763
172    30763
       ...  
444      166
332      166
316      166
150      166
395      166
Name: Distância (GCD), Length: 637, dtype: Int64


In [57]:
#salvar o df_viagens_unificadas
import os
local = f'dadosViagens/dados_viagens{ano}'
if not os.path.exists(local):
    print(f"📁 Diretório '{local}' não encontrado.")
df_viagens_unificadas.to_csv(f'{local}/viagens_unificadas_ufpb_{ano}.csv', index=False)
print(f"\n✅ DataFrame de Viagens Unificadas salvo como '{local}/viagens_unificadas_ufpb_{ano}.csv'.")


✅ DataFrame de Viagens Unificadas salvo como 'dadosViagens/dados_viagens2023/viagens_unificadas_ufpb_2023.csv'.


In [58]:
# carregar viagens_unificadas_ufpb
import pandas as pd
ano = 2023
local = f'dadosViagens/dados_viagens{ano}'  

viagens_unificadas_ufpb = pd.read_csv(f'{local}/viagens_unificadas_ufpb_{ano}.csv')

# comparar com os valores do trabalho anterior 

In [59]:
import pandas as pd
local_baseline = r"C:\Users\augusto\Documents\Portal da Transparência\2023_20240512_Viagens\Modelo 2023.2 Transparência.xlsx"
# --- CÉLULA DE COMPARAÇÃO (CORRIGIDA FINAL) ---
import pandas as pd

try:
    # 1. Carregar o DataFrame do Baseline (modelo feito à mão)
    df_baseline = pd.read_excel(local_baseline, sheet_name='Entradas')
    print("✅ Arquivo 'Modelo 2023.2 Transparência.xlsx - Entradas' carregado.")
except FileNotFoundError:
    print("⚠️ Arquivo Excel não encontrado, tentando carregar o CSV 'Entradas.csv'.")
    df_baseline = pd.read_csv('Modelo 2023.2 Transparência.xlsx - Entradas.csv')


    
# 2. Preparar os dados do seu modelo (UFPB)
#    #################################################################################
#    # CORREÇÃO PRINCIPAL: Seleciona a coluna 'Total_Emissoes_KgCO2eq'
#    #################################################################################
df_ufpb = df_viagens_unificadas[[
    'Identificador do processo de viagem',
    'Distância (GCD)',
    'Total_Emissoes_KgCO2eq'  # <-- ESTA É A MUDANÇA
]].copy()

# Renomeia as colunas para o merge
df_ufpb = df_ufpb.rename(columns={
    'Distância (GCD)': 'Distância (GCD)_UFPB',
    'Total_Emissoes_KgCO2eq': 'Emissões (KgCO2eq)_UFPB' # Renomeia a coluna correta
})


# 3. Preparar os dados do Baseline
df_baseline_clean = df_baseline.copy()
df_baseline_clean.columns = df_baseline_clean.columns.str.strip() # Limpa espaços nos nomes das colunas

df_baseline_clean = df_baseline_clean.rename(columns={
    'Distância (km)': 'Distância (GCD)_Baseline',
    'Emissões': 'Emissões (KgCO2eq)_Baseline'
})

# Agrupa para garantir que haja apenas uma linha por viagem
df_baseline_agg = df_baseline_clean.groupby('Identificador do processo de viagem').agg({
    'Distância (GCD)_Baseline': 'sum',
    'Emissões (KgCO2eq)_Baseline': 'sum'
}).reset_index()


# 4. Juntar (merge) os dois DataFrames
df_ufpb['Identificador do processo de viagem'] = df_ufpb['Identificador do processo de viagem'].astype(str)
df_baseline_agg['Identificador do processo de viagem'] = df_baseline_agg['Identificador do processo de viagem'].astype(str)

df_merged = pd.merge(
    df_ufpb,
    df_baseline_agg,
    on='Identificador do processo de viagem',
    how='inner'
)

# 5. Calcular as diferenças
df_merged['Diferença_Distância'] = df_merged['Distância (GCD)_UFPB'] - df_merged['Distância (GCD)_Baseline']
df_merged['Diferença_Emissões'] = df_merged['Emissões (KgCO2eq)_UFPB'] - df_merged['Emissões (KgCO2eq)_Baseline']

# 6. Salvar o resultado
colunas_finais = [
    'Identificador do processo de viagem', 'Distância (GCD)_UFPB', 'Distância (GCD)_Baseline',
    'Emissões (KgCO2eq)_UFPB', 'Emissões (KgCO2eq)_Baseline', 'Diferença_Distância', 'Diferença_Emissões'
]
df_resultado_final = df_merged[colunas_finais]

caminho_saida_diferenca = 'diferenca_FINAL_corrigida.csv'
df_resultado_final.to_csv(caminho_saida_diferenca, index=False)

print(f"\n✅ Comparação final concluída. Arquivo salvo em: '{caminho_saida_diferenca}'")
print("\n--- Amostra do Resultado Final ---")
print(df_resultado_final.head().to_string())

✅ Arquivo 'Modelo 2023.2 Transparência.xlsx - Entradas' carregado.

✅ Comparação final concluída. Arquivo salvo em: 'diferenca_FINAL_corrigida.csv'

--- Amostra do Resultado Final ---
  Identificador do processo de viagem  Distância (GCD)_UFPB  Distância (GCD)_Baseline  Emissões (KgCO2eq)_UFPB  Emissões (KgCO2eq)_Baseline  Diferença_Distância  Diferença_Emissões
0                            19281441                 13338               13338.20548                     2669                  2669.085364             -0.20548           -0.085364
1                            19280102                 13338               13338.20548                     2669                  2669.085364             -0.20548           -0.085364
2                            19281212                 13338               13338.20548                     2669                  2669.085364             -0.20548           -0.085364
3                            19281615                 13338               13338.20548       

In [60]:
# --- TESTE DE DIAGNÓSTICO 1: DISTÂNCIA TOTAL ---

viagem_id_distancia = '0000000000019281740'

# Pega os trechos individuais do DataFrame 'aereas_ufpb'
trechos_individuais = aereas_ufpb[
    aereas_ufpb['Identificador do processo de viagem'] == viagem_id_distancia
].drop_duplicates(subset=['Origem - Cidade', 'Destino - Cidade', 'Distância (GCD)'])

# Pega a viagem unificada final
viagem_unificada = df_viagens_unificadas[
    df_viagens_unificadas['Identificador do processo de viagem'] == viagem_id_distancia
]

print(f"--- Verificação da Viagem ID: {viagem_id_distancia} ---")

print("\nDistâncias dos Trechos Individuais Encontrados:")
print(trechos_individuais[['Origem - Cidade', 'Destino - Cidade', 'Distância (GCD)']].to_string(index=False))

print("\n--------------------------------------------------")
soma_manual_distancia = trechos_individuais['Distância (GCD)'].sum()
print(f"Soma manual dos trechos: {soma_manual_distancia} km")

if not viagem_unificada.empty:
    distancia_final = viagem_unificada['Distância (GCD)'].iloc[0]
    print(f"Distância Total na tabela final: {distancia_final} km")
    if soma_manual_distancia == distancia_final:
        print("\n✅ SUCESSO: A distância total na tabela final corresponde à soma dos trechos.")
    else:
        print("\n❌ FALHA: A distância total na tabela final NÃO corresponde à soma dos trechos.")
else:
    print("\n❌ FALHA: A viagem não foi encontrada no DataFrame final unificado.")

--- Verificação da Viagem ID: 0000000000019281740 ---

Distâncias dos Trechos Individuais Encontrados:
Empty DataFrame
Columns: [Origem - Cidade, Destino - Cidade, Distância (GCD)]
Index: []

--------------------------------------------------
Soma manual dos trechos: 0.0 km

❌ FALHA: A viagem não foi encontrada no DataFrame final unificado.


In [61]:
# --- TESTE DE DIAGNÓSTICO 2: CÁLCULO DE EMISSÃO ---

viagem_id_emissao = '0000000000019375473'

viagem_unificada_emissao = df_viagens_unificadas[
    df_viagens_unificadas['Identificador do processo de viagem'] == viagem_id_emissao
]

print(f"--- Verificação da Viagem ID: {viagem_id_emissao} ---")

if not viagem_unificada_emissao.empty:
    distancia_total = viagem_unificada_emissao['Distância (GCD)'].iloc[0]
    emissao_total = viagem_unificada_emissao['Total_Emissoes_KgCO2eq'].iloc[0]
    
    # Fator de emissão para Longa Distância (> 3700 km)
    EF_LONGA_DISTANCIA = 0.200108215
    
    emissao_esperada = round(distancia_total * EF_LONGA_DISTANCIA)
    
    print(f"Distância Total da Viagem: {distancia_total} km")
    print(f"Emissão Total na Tabela Final: {emissao_total} kgCO2eq")
    print("--------------------------------------------------")
    print(f"Cálculo Esperado (Distância * Fator): {distancia_total} * {EF_LONGA_DISTANCIA:.4f} ≈ {emissao_esperada} kgCO2eq")
    
    # Compara o valor calculado com o esperado, com uma pequena margem de tolerância para arredondamento
    if abs(emissao_total - emissao_esperada) <= 1:
        print("\n✅ SUCESSO: O valor da emissão total está consistente com a distância e o fator de emissão.")
    else:
        print("\n❌ FALHA: O valor da emissão total está inconsistente. O bug da 'metade da emissão' pode persistir.")
else:
    print("\n❌ FALHA: A viagem não foi encontrada no DataFrame final unificado.")

--- Verificação da Viagem ID: 0000000000019375473 ---

❌ FALHA: A viagem não foi encontrada no DataFrame final unificado.


In [62]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configurações gerais ---
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# --- Histograma das diferenças de distância ---
plt.figure()
sns.histplot(comparacao["Diferença_Distância"], bins=50, kde=True, color="skyblue")
plt.title("Distribuição das Diferenças de Distância (UFPB - Baseline)")
plt.xlabel("Diferença de Distância (km)")
plt.ylabel("Número de Viagens")
plt.axvline(0, color='red', linestyle='--', label='Sem diferença')
plt.legend()
plt.show()

# --- Histograma das diferenças de emissões ---
plt.figure()
sns.histplot(comparacao["Diferença_Emissões"], bins=50, kde=True, color="lightgreen")
plt.title("Distribuição das Diferenças de Emissões (UFPB - Baseline)")
plt.xlabel("Diferença de Emissões (KgCO2eq)")
plt.ylabel("Número de Viagens")
plt.axvline(0, color='red', linestyle='--', label='Sem diferença')
plt.legend()
plt.show()

# --- Boxplots para identificar outliers ---
plt.figure()
sns.boxplot(data=comparacao[["Diferença_Distância", "Diferença_Emissões"]])
plt.title("Boxplot das Diferenças de Distância e Emissões")
plt.ylabel("Valor da Diferença")
plt.show()


NameError: name 'comparacao' is not defined

<Figure size 1200x600 with 0 Axes>

In [ ]:
comparacao[["Identificador do processo de viagem","Diferença_Distância", "Diferença_Emissões"]]


,Identificador do processo de viagem,Diferença_Distância,Diferença_Emissões
0,19281441,0,-1335
1,19280102,0,-1335
2,19281212,0,-1335
3,19281615,0,-1335
4,19281740,0,-1335
...,...,...,...
409,19065213,-3097,-575
410,19253229,-3097,-575
411,19245814,-3097,-575
412,19062585,-3097,-575


In [ ]:
outliers = comparacao[(comparacao["Diferença_Distância"].abs() > 500) | 
                      (comparacao["Diferença_Emissões"].abs() > 500)]
outliers


,Identificador do processo de viagem,Distância (GCD)_UFPB,Emissões (KgCO2eq)_UFPB,Distância (GCD)_Baseline,Emissões (KgCO2eq)_Baseline,Diferença_Distância,Diferença_Emissões
0,19281441,13338,1334,13338.205480,2669.085364,0,-1335
1,19280102,13338,1334,13338.205480,2669.085364,0,-1335
2,19281212,13338,1334,13338.205480,2669.085364,0,-1335
3,19281615,13338,1334,13338.205480,2669.085364,0,-1335
4,19281740,13338,1334,13338.205480,2669.085364,0,-1335
...,...,...,...,...,...,...,...
409,19065213,166,22,3262.912258,596.686658,-3097,-575
410,19253229,166,22,3262.912258,596.686658,-3097,-575
411,19245814,166,22,3262.912258,596.686658,-3097,-575
412,19062585,166,22,3262.912258,596.686658,-3097,-575


# calculo das metricas 

In [ ]:
import numpy as np
import pandas as pd

# --- 1. PREPARAÇÃO: GERAÇÃO DA TABELA DE RESULTADOS (metrics_df) usando aereas_ufpb ---
# O aereas_ufpb já é o DataFrame de viagens aéreas únicas e tem as colunas agregadas.

total_viagens = len(aereas_ufpb)
urgentes_sem_justificativa = aereas_ufpb['Viagem_Urgente_Sem_Justificativa'].sum()

metrics_df = pd.DataFrame({
    'Métrica': [
        'Total Viagens (ED2.3)', 
        'Total Emissões (KgCO2eq, ED1.1)', 
        'Total Custo (R$)', 
        'Viagens Evitáveis (Qtd, <= 600km)', 
        'Viagens Urgentes S/ Just. (DF1.5 Qtd)'
    ],
    'Valor': [
        total_viagens,
        aereas_ufpb['Total_Emissoes_KgCO2eq'].sum(),
        aereas_ufpb['Total_Custo_R$'].sum(),
        aereas_ufpb['Teve_Trecho_Evitavel'].sum(),
        urgentes_sem_justificativa
    ]
})

print("✅ Geração da Tabela de Métricas (metrics_df) concluída para UFPB.")
print("\n--- Métricas Chave do Modelo (UFPB) ---")
print(metrics_df.to_string(index=False))


# --- 2. CÁLCULO DO INDICADOR DF1.5 SCORE ---
if total_viagens > 0:
    df1_5_proporcao = urgentes_sem_justificativa / total_viagens
    df1_5_score = 1.0 - df1_5_proporcao
else:
    df1_5_proporcao = 0.0
    df1_5_score = 1.0 

# Tabela de Scores
scores_df = pd.DataFrame({
    'Indicador': ['ED2.3 (Total Viagens)', 'DF1.5 (Proporção)', 'DF1.5 (Score)'],
    'Valor': [total_viagens, df1_5_proporcao, df1_5_score]
})

print("\n✅ Scores (DF1.5 e ED2.3) calculados para a UFPB.")

✅ Geração da Tabela de Métricas (metrics_df) concluída para UFPB.

--- Métricas Chave do Modelo (UFPB) ---
                              Métrica        Valor
                Total Viagens (ED2.3) 1.287000e+03
      Total Emissões (KgCO2eq, ED1.1) 1.099447e+06
                     Total Custo (R$) 5.015413e+06
    Viagens Evitáveis (Qtd, <= 600km) 4.280000e+02
Viagens Urgentes S/ Just. (DF1.5 Qtd) 0.000000e+00

✅ Scores (DF1.5 e ED2.3) calculados para a UFPB.


In [ ]:
# # --- CÉLULA 7: GERAÇÃO DE MÉTRICAS E SCORE DF1.5 ---

# # Agora 'aereas' é o DataFrame de viagens únicas.
# total_viagens = len(aereas)
# urgentes_sem_justificativa = aereas['Viagem_Urgente_Sem_Justificativa'].sum()

# # 1. GERAÇÃO DA TABELA DE MÉTRICAS BRUTAS
# metrics_df = pd.DataFrame({
#     'Métrica': [
#         'Total Viagens (ED2.3)', 
#         'Total Emissões (KgCO2eq, ED1.1)', 
#         'Total Custo (R$)', 
#         'Viagens Evitáveis (Qtd, <= 600km)', 
#         'Viagens Urgentes S/ Just. (DF1.5 Qtd)'
#     ],
#     'Valor': [
#         total_viagens,
#         aereas['Total_Emissoes_KgCO2eq'].sum(),
#         aereas['Total_Custo_R$'].sum(),
#         aereas['Teve_Trecho_Evitavel'].sum(),
#         urgentes_sem_justificativa
#     ]
# })

# # 2. CÁLCULO DO INDICADOR DF1.5 SCORE
# if total_viagens > 0:
#     df1_5_proporcao = urgentes_sem_justificativa / total_viagens
#     df1_5_score = 1.0 - df1_5_proporcao
# else:
#     df1_5_proporcao = 0.0
#     df1_5_score = 1.0 

# # Tabela de Scores
# scores_df = pd.DataFrame({
#     'Indicador': ['ED2.3 (Total Viagens)', 'DF1.5 (Proporção)', 'DF1.5 (Score)'],
#     'Valor': [total_viagens, df1_5_proporcao, df1_5_score]
# })

# print("✅ Geração da Tabela de Métricas (metrics_df) e Scores (scores_df) concluída.")
# print("\n--- Scores Chave ---")
# print(scores_df.to_string(index=False))

In [ ]:
# --- CÉLULA 7b: DEFINIÇÃO DE CONSTANTES DO MODELO (Hardcoded Baseline) ---
# Estes valores são extraídos da Baseline Oficial do modelo (Baseline 19.2 - 22.2) 
# e são usados como o ponto de comparação fixo para o Score, eliminando a dependência do arquivo Excel.

# Baseline ED1.1 (Emissões Totais em KgCO2eq)
BASELINE_ED1_1 = 285440.32313309913 

# Baseline ED2.1 (Custo Total em R$)
BASELINE_ED2_1 = 1610128.7746285002 

print("✅ Constantes de Baseline ED1.1 e ED2.1 definidas para cálculo independente.")

✅ Constantes de Baseline ED1.1 e ED2.1 definidas para cálculo independente.


In [ ]:
# --- CÉLULA 8: CÁLCULO DO SCORE ED1.1 (EMISSÕES) - MODIFICADA ---

# 1. Parâmetros (Não precisamos mais do caminho do Excel)
COLUNA_BASELINE = 'Baseline (19.2 - 22.2)' # Apenas um rótulo de exibição
INDICADOR_ALVO = 'ED1.1'

# 2. Extrair a Baseline ED1.1 (Usando a constante)
baseline_ed1_1 = BASELINE_ED1_1 # Definido na Célula 7b

# Extrai o valor atual de Emissões (da Célula 7)
metrica_atual_emissao = metrics_df[metrics_df['Métrica'] == 'Total Emissões (KgCO2eq, ED1.1)']['Valor'].iloc[0]


# 3. CÁLCULO DO SCORE ED1.1
variacao = (metrica_atual_emissao - baseline_ed1_1) / baseline_ed1_1
score_ed1_1 = 0.0

if metrica_atual_emissao <= baseline_ed1_1:
    score_ed1_1 = 1.0
else:
    score_ed1_1 = max(0, 1.0 - (variacao * 2)) 


# 4. CONSOLIDAÇÃO (Adiciona os resultados ao scores_df)
scores_df.loc[len(scores_df)] = ['ED1.1 (Baseline)', baseline_ed1_1]
scores_df.loc[len(scores_df)] = ['ED1.1 (Variação %)', variacao]
scores_df.loc[len(scores_df)] = ['ED1.1 (Score)', score_ed1_1]


print("✅ Cálculo do Score ED1.1 concluído.")
print(f"\nBaseline ED1.1 Utilizada: {baseline_ed1_1:.2f} KgCO2eq")
print(f"Emissão Atual: {metrica_atual_emissao:.2f} KgCO2eq")
print(f"Variação: {variacao * 100:.2f}%")
print("\n--- Scores Atualizados ---")
print(scores_df.to_string(index=False))

✅ Cálculo do Score ED1.1 concluído.

Baseline ED1.1 Utilizada: 285440.32 KgCO2eq
Emissão Atual: 1099447.22 KgCO2eq
Variação: 285.18%

--- Scores Atualizados ---
            Indicador         Valor
ED2.3 (Total Viagens)   1287.000000
    DF1.5 (Proporção)      0.000000
        DF1.5 (Score)      1.000000
     ED1.1 (Baseline) 285440.323133
   ED1.1 (Variação %)      2.851759
        ED1.1 (Score)      0.000000


In [ ]:
# --- CÉLULA 9: CÁLCULO DO SCORE ED2.1 (CUSTO TOTAL) - MODIFICADA ---

# 1. Parâmetros e Baseline (Usando a constante)
COLUNA_BASELINE = 'Baseline (19.2 - 22.2)' 
INDICADOR_ALVO = 'ED2.1'

# Extrai a Baseline ED2.1 (Usando a constante)
baseline_ed2_1 = BASELINE_ED2_1 # Definido na Célula 7b

# Extrai a Métrica Atual (Custo Total R$)
metrica_atual_custo = metrics_df[metrics_df['Métrica'] == 'Total Custo (R$)']['Valor'].iloc[0]


# 2. CÁLCULO DO SCORE ED2.1
variacao_custo = (metrica_atual_custo - baseline_ed2_1) / baseline_ed2_1
score_ed2_1 = 0.0

if metrica_atual_custo <= baseline_ed2_1:
    score_ed2_1 = 1.0
else:
    # Decaimento do score (usando o mesmo multiplicador de 2)
    score_ed2_1 = max(0, 1.0 - (variacao_custo * 2)) 


# 3. CONSOLIDAÇÃO 
scores_df.loc[len(scores_df)] = ['ED2.1 (Baseline)', baseline_ed2_1]
scores_df.loc[len(scores_df)] = ['ED2.1 (Variação %)', variacao_custo]
scores_df.loc[len(scores_df)] = ['ED2.1 (Score)', score_ed2_1]


print("✅ Cálculo do Score ED2.1 (Custo Total) concluído.")
print(f"\nBaseline ED2.1 Utilizada: {baseline_ed2_1:.2f} R$")
print(f"Custo Atual: {metrica_atual_custo:.2f} R$")
print(f"Variação: {variacao_custo * 100:.2f}%")
print("\n--- Scores Atualizados ---")
print(scores_df.to_string(index=False))

✅ Cálculo do Score ED2.1 (Custo Total) concluído.

Baseline ED2.1 Utilizada: 1610128.77 R$
Custo Atual: 5015413.00 R$
Variação: 211.49%

--- Scores Atualizados ---
            Indicador        Valor
ED2.3 (Total Viagens) 1.287000e+03
    DF1.5 (Proporção) 0.000000e+00
        DF1.5 (Score) 1.000000e+00
     ED1.1 (Baseline) 2.854403e+05
   ED1.1 (Variação %) 2.851759e+00
        ED1.1 (Score) 0.000000e+00
     ED2.1 (Baseline) 1.610129e+06
   ED2.1 (Variação %) 2.114914e+00
        ED2.1 (Score) 0.000000e+00


In [ ]:
import pandas as pd
import os
from datetime import datetime

# --- PARÂMETROS DE SALVAMENTO ---
# Nome do arquivo de saída. Adiciona timestamp para evitar sobrescrever dad
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
orgao = 'UFPB'  # Nome do órgão para o arquivo
NOME_ARQUIVO_SAIDA = f"dadosViagens/dados_viagens{ano}/relatorio_metrica_score_{orgao}_{ano}_{TIMESTAMP}.csv"
CAMINHO_SAIDA = os.path.join(os.getcwd(), NOME_ARQUIVO_SAIDA)

# 1. PREPARAÇÃO E RENOMEAÇÃO (metrics_df)
# Prepara a tabela de métricas brutas
metrics_df_clean = metrics_df.rename(columns={'Métrica': 'Indicador', 'Valor': 'Métrica Bruta'})
metrics_df_clean['Tipo'] = 'Métrica Bruta (Qtd/Valor)'


# 2. PREPARAÇÃO E RENOMEAÇÃO (scores_df)
# Prepara a tabela de scores
scores_df_clean = scores_df.rename(columns={'Valor': 'Score/Baseline'})
scores_df_clean['Tipo'] = 'Score/Baseline'


# 3. CONCATENAÇÃO FINAL
# Junta os dois DataFrames para um relatório unificado
relatorio_final_df = pd.concat([metrics_df_clean, scores_df_clean], ignore_index=True)


# 4. SALVAMENTO
try:
    relatorio_final_df.to_csv(
        CAMINHO_SAIDA, 
        index=False, 
        sep=';', # Usa ponto e vírgula para compatibilidade com Excel
        encoding='latin1'
    )
    print(f"✅ Sucesso! Relatório final salvo em:\n{CAMINHO_SAIDA}")

except Exception as e:
    print(f"❌ ERRO ao salvar o arquivo: {e}")

✅ Sucesso! Relatório final salvo em:
d:\projetoExtensao\viagensAServico\dadosViagens/dados_viagens2023/relatorio_metrica_score_UFPB_2023_20251022_125706.csv
